In [1]:
import pandas as pd
import sqlite3
import os
from datetime import datetime

In [ ]:
CSV = "g61_ELearning_Certificates_v7_4.csv"


df = pd.read_csv(
    CSV,
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
    quotechar='"',
    engine="python",
)
df.columns = [c.strip() for c in df.columns]

print(f"Linhas: {len(df)}  |  Colunas: {len(df.columns)}")
print(f"Plataformas: {df['platform_id'].nunique()}  |  "
      f"Cursos: {df['course_name'].nunique()}  |  "
      f"Estudantes: {df['student_id'].nunique()}")

In [ ]:
def to_iso(s):
    s = str(s).strip()
    if s in ("", "nan", "None"):
        return None
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(s, fmt).date().isoformat()
        except ValueError:
            continue
    return None

def to_fee(s):
    s = str(s).strip().replace(",", ".")
    if s in ("", "nan", "None"):
        return None
    try:
        return round(float(s), 2)
    except ValueError:
        return None

to_int = lambda col: pd.to_numeric(col, errors="coerce").astype("Int64")

df["issue_date"]      = df["issue_date"].map(to_iso)
df["online_date"]     = df["online_date"].map(to_iso)
df["certificate_fee"] = df["certificate_fee"].map(to_fee)


df["course_info"] = df["course_info"].str.strip().str.replace(";", ",", regex=False)


name2id = (df[df["course_id"].str.strip() != ""]
           .drop_duplicates("course_name")
           .set_index("course_name")["course_id"].to_dict())
df["course_id"] = df.apply(
    lambda r: r["course_id"] if str(r["course_id"]).strip()
              else name2id.get(r["course_name"], ""),
    axis=1)

df["certificate_name"] = df["certificate_name"].apply(
    lambda x: x if str(x).strip() else None)

print("Limpeza concluida.")

In [ ]:

df_platform = (df[["platform_id", "platform_name", "platform_country"]]
               .assign(platform_id=lambda d: to_int(d["platform_id"]))
               .dropna(subset=["platform_id"])
               .drop_duplicates("platform_id")
               .sort_values("platform_id")
               .rename(columns={"platform_id": "id"})
               .reset_index(drop=True))


df_certificate = pd.DataFrame({
    "id"                  : to_int(df["certificate_id"]),
    "certificate_name"    : df["certificate_name"],
    "certificate_type"    : df["certificate_type"],
    "certificate_language": df["certificate_language"],
    "issue_date"          : df["issue_date"],
}).reset_index(drop=True)


df_course = (df[["course_id", "course_name", "course_category", "course_info",
                 "online_date", "platform_id"]]
             .assign(course_id=lambda d: to_int(d["course_id"]),
                     platform_id=lambda d: to_int(d["platform_id"]))
             .dropna(subset=["course_id"])
             .rename(columns={"course_id": "id"})
             .reset_index(drop=True))


df_student = (df[["student_id", "student_name", "student_email",
                  "student_country", "student_age"]]
              .assign(student_id=lambda d: to_int(d["student_id"]))
              .dropna(subset=["student_id"])
              .sort_values("student_id")
              .groupby("student_id", as_index=False).first()
              .rename(columns={"student_id": "id"}))
df_student["student_age"] = to_int(df_student["student_age"])


df_trans = pd.DataFrame({
    "id"             : to_int(df["transaction_id"]),
    "platform_id"    : to_int(df["platform_id"]),
    "certificate_id" : to_int(df["certificate_id"]),
    "student_id"     : to_int(df["student_id"]),
    "certificate_fee": df["certificate_fee"],
    "payment_method" : df["payment_method"],
})

print(f"Platform:    {len(df_platform):>6}")
print(f"Certificate: {len(df_certificate):>6}")
print(f"Course:      {len(df_course):>6}")
print(f"Student:     {len(df_student):>6}")
print(f"Trans:       {len(df_trans):>6}")

In [ ]:

os.makedirs("data", exist_ok=True)

con = sqlite3.connect("data/novadatabase6.db")
cur = con.cursor()

for t in ["Platform", "Certificate", "Course", "Student", "Trans"]:
    cur.execute(f"DROP TABLE IF EXISTS {t}")

cur.execute("CREATE TABLE Platform (id INTEGER, platform_name TEXT, platform_country TEXT)")
cur.execute("""CREATE TABLE Certificate (id INTEGER, certificate_name TEXT,
               certificate_type TEXT, certificate_language TEXT, issue_date DATE)""")
cur.execute("""CREATE TABLE Course (id INTEGER, course_name TEXT, course_category TEXT,
               course_info TEXT, online_date DATE, platform_id INTEGER)""")
cur.execute("""CREATE TABLE Student (id INTEGER, student_name TEXT, student_email TEXT,
               student_country TEXT, student_age INTEGER)""")
cur.execute("""CREATE TABLE Trans (id INTEGER, platform_id INTEGER, certificate_id INTEGER,
               student_id INTEGER, certificate_fee REAL, payment_method TEXT)""")
con.commit()

df_platform.to_sql("Platform",       con, if_exists="append", index=False)
df_certificate.to_sql("Certificate", con, if_exists="append", index=False)
df_course.to_sql("Course",           con, if_exists="append", index=False)
df_student.to_sql("Student",         con, if_exists="append", index=False)
df_trans.to_sql("Trans",             con, if_exists="append", index=False)
con.commit()
con.close()
print("Base de dados 'data/novadatabase6.db' criada com sucesso!")

In [ ]:
con = sqlite3.connect("data/novadatabase6.db")
for t in ["Platform", "Certificate", "Course", "Student", "Trans"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:12} {n:>6} registos")

print("\nNomes de certificado distintos:",
      con.execute("SELECT COUNT(DISTINCT certificate_name) FROM Certificate "
                  "WHERE certificate_name IS NOT NULL").fetchone()[0])
print("Linguas por certificado (exemplo):")
for row in con.execute("""SELECT certificate_name, GROUP_CONCAT(DISTINCT certificate_language)
                          FROM Certificate WHERE certificate_name IS NOT NULL
                          GROUP BY certificate_name LIMIT 5"""):
    print("  ", row[0], "->", row[1])
con.close()